# 02 · Reproducibility — Cheap vs Flagship *and* the working-with-reasoning-models reference

This notebook now plays **two roles at once**:

1. **A disciplined cheap-vs-flagship reproducibility experiment.** The claim we test: a *cheap* reasoning model (**DeepSeek V4 Pro**, `deepseek-v4-pro`) can **match a flagship** (Claude Opus 5 / GPT-5.6) on a **specific narrow task** — at a fraction of the cost. We add discipline with a fixed eval set, an LLM-as-judge, fixed seeds, and `temperature=0`, so the comparison is *repeatable* rather than anecdotal.
2. **The hands-on "working with reasoning models in practice" reference** (the role formerly split with the now-archived NB04). The live adapters below show the **current** provider mechanics an engineer actually needs: OpenAI Responses API with `reasoning.effort` + a **developer** message; Anthropic **adaptive thinking** (`thinking={"type":"adaptive"}` + `output_config.effort`, *not* the deprecated `budget_tokens`); reasoning/thinking tokens read from `usage.*_tokens_details` and **billed as output tokens**; and a **truncation guard** for incomplete responses. It closes with a production best-practices checklist (§14) mirroring deck Section 3.

> **Punchline for students:** don't reflexively reach for the most expensive model. Pick **per task**, instrument the *hidden* reasoning-token cost, reserve token headroom, and turn your "vibe check" into something you can re-run tomorrow and get (nearly) the same answer.

---

The model registry, provider adapters, and LLM-as-judge are **self-contained here** (they used to be shared with `04_picking_a_reasoning_model_for_an_application.ipynb`, now archived). A deterministic **mock-mode** lets the whole notebook run **with no API keys at all**.

| | |
|---|---|
| **Cheap model** | `deepseek-v4-pro` (OpenAI-compatible chat, `seed=` supported) |
| **Flagship** | `claude-opus-5` (preferred) → falls back to `gpt-5.6` |
| **Eval set** | 10 self-contained reasoning prompts with reference answers |
| **Judge** | OpenAI → Anthropic → keyword fallback |
| **Reasoning-token cost surfaced?** | **Yes** — per-call `reasoning_tokens`, billed as output |
| **Runs with no keys?** | **Yes** — deterministic seeded mock mode |

## 0 · Setup & environment

```bash
uv pip install openai anthropic pandas matplotlib numpy python-dotenv
```

Keys go in a `.env` file (only the ones you have — everything else is mocked):

```
OPENAI_API_KEY=...
ANTHROPIC_API_KEY=...
DEEPSEEK_API_KEY=...
```

In [ ]:
import os, json, time, random, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()

# ---------------------------------------------------------------- SEED everything
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def have(*names):
    """Return the first non-empty env var among names (or None)."""
    for n in names:
        v = os.environ.get(n)
        if v:
            return v
    return None

print("-" * 60)
print("Keys detected:")
for label, names in [
    ("OpenAI",   ["OPENAI_API_KEY"]),
    ("Anthropic",["ANTHROPIC_API_KEY"]),
    ("DeepSeek", ["DEEPSEEK_API_KEY"]),
]:
    print(f"  {label:10s}: {'set' if have(*names) else '-- missing (will MOCK)'}")
print("-" * 60)
print(f"Global SEED = {SEED}  (random + numpy seeded)")

## 1 · Model registry

Each entry says which API style it uses, the model id, the base URL (for OpenAI-compatible endpoints), and which env var holds the key. We only need three of the five models here, but we keep the full registry so you can swap contenders freely.

In [ ]:
MODELS = {
    "gpt-5.6": {
        "label": "GPT-5.6",
        "kind": "openai_responses",
        "model_id": "gpt-5.6",
        "keys": ["OPENAI_API_KEY"],
    },
    "claude-opus-5": {
        "label": "Claude Opus 5",
        "kind": "anthropic",
        "model_id": "claude-opus-5",
        "keys": ["ANTHROPIC_API_KEY"],
    },
    "deepseek-v4-pro": {
        "label": "DeepSeek V4 Pro",
        "kind": "chat",                       # OpenAI-compatible
        "model_id": "deepseek-v4-pro",
        "base_url": "https://api.deepseek.com",
        "keys": ["DEEPSEEK_API_KEY"],
    },
}

ACTIVE = [k for k, s in MODELS.items() if have(*s["keys"])]
print("Models with a live key:", ACTIVE or "(none -- everything runs in MOCK mode)")

# ---------------------------------------------------------------- the matchup
CHEAP_KEY = "deepseek-v4-pro"
# Prefer Opus 5 as the flagship; fall back to GPT-5.6 if only that key is present
# (and fall back to it anyway in pure-mock mode so the story is consistent).
if have(*MODELS["claude-opus-5"]["keys"]):
    FLAGSHIP_KEY = "claude-opus-5"
elif have(*MODELS["gpt-5.6"]["keys"]):
    FLAGSHIP_KEY = "gpt-5.6"
else:
    FLAGSHIP_KEY = "claude-opus-5"   # mock default

CONTENDERS = [CHEAP_KEY, FLAGSHIP_KEY]
print(f"Cheap contender : {MODELS[CHEAP_KEY]['label']}  ({CHEAP_KEY})")
print(f"Flagship        : {MODELS[FLAGSHIP_KEY]['label']}  ({FLAGSHIP_KEY})")

## 2 · Cost table — **editable 2026 placeholder estimates**

These are **illustrative** USD prices per **million tokens** (input / output). They are *not* live pricing — update them from each provider's pricing page on the day you run a real comparison. They exist so the notebook can compute an *estimated* cost per prompt.

> **The key cost fact (deck §3):** a reasoning model's hidden *thinking* tokens are **billed as output tokens** on every provider. So the `out` price below is what you pay for reasoning too, and a call's real cost is *output-token-dominated*. The results table surfaces a per-call `reasoning_tokens` count so you can see the hidden spend — the lesson is **cost per task, not per token**.

In [ ]:
# !!! EDITABLE PLACEHOLDER PRICING -- replace with real numbers before any real decision.
COST_PER_MTOK = {
    # model_key            input $/Mtok   output $/Mtok
    "gpt-5.6":          {"in": 2.50, "out": 15.00},
    "claude-opus-5":  {"in": 5.00, "out": 25.00},
    "deepseek-v4-pro":  {"in": 0.28, "out": 0.42},   # ~10-50x cheaper -- the whole point
}

def estimate_cost(model_key, in_tokens, out_tokens):
    p = COST_PER_MTOK[model_key]
    return (in_tokens / 1e6) * p["in"] + (out_tokens / 1e6) * p["out"]

print("Placeholder pricing (USD per million tokens) -- EDIT THESE:")
print("-" * 60)
for k in CONTENDERS:
    p = COST_PER_MTOK[k]
    print(f"  {MODELS[k]['label']:18s}  in=${p['in']:>6.2f}/Mtok   out=${p['out']:>6.2f}/Mtok")
print("-" * 60)
ratio = (COST_PER_MTOK[FLAGSHIP_KEY]['out'] / COST_PER_MTOK[CHEAP_KEY]['out'])
print(f"Headline output-price ratio (flagship / cheap): ~{ratio:.0f}x")

## 3 · The vibe-check task + a fixed eval set

**Task:** short, self-contained reasoning questions about reasoning-model / LLM-training concepts (the same subject matter as the `notes_dataset/` used in notebook 04), each with a concise **reference answer**. This is exactly the kind of *narrow, well-specified* task where a cheap model often matches a flagship.

The eval set is a **fixed, ordered list** — we never shuffle it — which is one of the things that makes the comparison reproducible.

In [ ]:
EVAL_SET = [
    {"id": "q01",
     "question": "In GRPO, how is the advantage for each completion computed, and which network does it remove compared to PPO?",
     "reference": "The advantage is group-relative: (reward - group mean) / group std over a sampled group of completions. It removes the value/critic network that PPO must train."},
    {"id": "q02",
     "question": "What problem do DAPO and Dr. GRPO both fix in GRPO?",
     "reference": "They both fix the length-normalization bias (the 1/|o| per-output factor) that makes the model prefer longer incorrect outputs. DAPO normalizes over total group tokens (plus clip-higher); Dr. GRPO drops the length term entirely."},
    {"id": "q03",
     "question": "How many stages are in the DeepSeek R1 training pipeline, and what is stage 2?",
     "reference": "Five stages. Stage 2 is cold-start SFT: a few thousand cleaned long chain-of-thought examples used as an anchor before RL."},
    {"id": "q04",
     "question": "What reward does DeepSeek R1-Zero use, and what two delimiters does its prompt template require?",
     "reference": "A verifiable format + accuracy reward (no learned reward model). The template puts reasoning in <think></think> and the final answer in <answer></answer>."},
    {"id": "q05",
     "question": "Give the two standard intuitions for why chain-of-thought helps.",
     "reference": "(1) Decomposition: splitting a hard problem into sub-problems reuses seen patterns. (2) More test-time compute: each emitted token is another forward pass, so more reasoning tokens buy more computation before answering."},
    {"id": "q06",
     "question": "Name two techniques for controlling a reasoning model's thinking budget.",
     "reference": "Any two of: dynamic budget routing, budget forcing ('Wait' to extend / 'time is up' to stop), context-aware budgeting, or continuous/latent thoughts (Coconut)."},
    {"id": "q07",
     "question": "Per the o1 prompting guidance, should you tell a reasoning model to 'think step by step', and what replaces system messages?",
     "reference": "No -- explicit 'think step by step' prompts are unnecessary and can hurt. Developer messages replace system messages for these models."},
    {"id": "q08",
     "question": "What does the GSM-Symbolic study conclude, including the effect of irrelevant information?",
     "reference": "LLM math reasoning is fragile and pattern-matching-like: accuracy varies with numeric changes and drops as clauses increase. Irrelevant info (GSM-NoOp) can cut performance by up to ~65%."},
    {"id": "q09",
     "question": "In R1-style reasoning distillation, is the student trained by logit matching or plain SFT, and how does it compare to RL-from-scratch at small scale?",
     "reference": "Plain SFT on the teacher's generated reasoning traces (not logit matching). At small scale, distillation beats RL-from-scratch (e.g. distilled Qwen-32B ~72.6 AIME vs ~47.0 from RL)."},
    {"id": "q10",
     "question": "What does the KL penalty enforce in RL fine-tuning, and where does the KL term live in PPO vs GRPO?",
     "reference": "It keeps the policy near a trusted reference (the SFT model). PPO folds KL into the per-token reward before advantage estimation; GRPO uses a separate explicit -beta*KL term in the objective."},
]

# Fix the order explicitly (reproducibility): sort by id, never shuffle.
EVAL_SET = sorted(EVAL_SET, key=lambda t: t["id"])
print(f"{len(EVAL_SET)} evaluation prompts (fixed order):")
for t in EVAL_SET:
    print(f"  {t['id']}: {t['question'][:70]}...")

## 4 · What is — and isn't — reproducible across hosted models

We pin everything we *can* control:

- **`temperature = 0`** (lowest / greedy decoding) for every provider that accepts it.
- **`seed=SEED`** passed to providers that accept it — OpenAI Responses and DeepSeek/OpenAI-compatible chat both take a `seed`. Anthropic does **not** expose a seed (and Opus 5 rejects an explicit `temperature`), so for Opus we rely on its **default greedy decoding** for determinism.
- **Python / NumPy seeds** fixed, and a **fixed eval-set order** (sorted by id, never shuffled).
- The **judge** also runs at `temperature=0` with low reasoning effort.

**Honest caveat:** hosted reasoning models are **not bit-exact**, even at `temperature=0` with a seed. Floating-point non-determinism, server-side batching, mixture-of-experts routing, and silent model updates all introduce variance. Seeds + a fixed eval set **reduce** run-to-run drift and make a comparison *defensible*; they do not make it *identical*. Treat the seed as "shrink the noise", not "eliminate it". The mock mode below, by contrast, **is** perfectly deterministic — useful precisely because it isolates the harness from provider noise.

## 5 · Provider adapters (live) — current reasoning-model mechanics, pinned for reproducibility

Single-shot adapters, **pinned** (seed + `temperature=0` where accepted) and updated to the mechanics in `research/working_with_reasoning_models_practical.md`. They are only ever called when a real key is present; otherwise the mock path in §6 runs instead. What each adapter gets right:

- **OpenAI (Responses API):** `reasoning={"effort": "low"}`, a **developer** message (not `system` — required from o-series/GPT-5 onward), reasoning cost read from `usage.output_tokens_details.reasoning_tokens`, and a **truncation guard** (`status == "incomplete"` / `incomplete_details.reason == "max_output_tokens"`).
- **Anthropic (Messages API):** **adaptive thinking** (`thinking={"type": "adaptive"}`) with depth via `output_config={"effort": "low"}` — *not* the deprecated `budget_tokens`, which is rejected with a 400 on the newest models. Thinking cost read from `usage.output_tokens_details.thinking_tokens`; no explicit `temperature` (Opus 5 rejects a non-default one). Truncation checked via `stop_reason`.
- **DeepSeek (OpenAI-compatible chat):** `seed` + `temperature=0`; `finish_reason == "length"` flags truncation.

Every adapter returns the **same shape** — including `reasoning_tokens` and a `truncated` flag — so cost accounting and the truncation guard work identically live or mocked. Reasoning/thinking tokens are **included in `out_tokens`**, so `estimate_cost` already bills them as output.

In [ ]:
# Direct, goal-oriented prompt -- deliberately NO "think step by step".
# Reasoning models reason internally; over-prompting CoT hurts (research doc, deck S9).
SYSTEM_PROMPT = (
    "You are a precise assistant answering questions about reasoning models and "
    "LLM training. Answer in 2-4 concise sentences. State the key facts directly."
)

# Build live clients only for models that actually have a key ------------------
CLIENTS = {}
_openai = None
if have("OPENAI_API_KEY"):
    from openai import OpenAI
    _openai = OpenAI()

for key in ACTIVE:
    spec = MODELS[key]
    kind = spec["kind"]
    if kind == "openai_responses":
        CLIENTS[key] = _openai
    elif kind == "anthropic":
        from anthropic import Anthropic
        CLIENTS[key] = Anthropic()
    elif kind == "chat":
        from openai import OpenAI
        CLIENTS[key] = OpenAI(api_key=have(*spec["keys"]), base_url=spec["base_url"])

print("Live clients built for:", list(CLIENTS) or "(none)")


# --- OpenAI Responses API (gpt-5.6) ------------------------------------------
#   * developer message (NOT system) -- required from o-series / GPT-5 onward
#   * reasoning.effort enum; reasoning_tokens read from usage details
#   * truncation guard: status == "incomplete" / reason == "max_output_tokens"
def run_openai(client, model_id, question):
    t0 = time.perf_counter()
    resp = client.responses.create(
        model=model_id, reasoning={"effort": "low"},
        temperature=0, seed=SEED,
        max_output_tokens=2048,                 # reserve headroom for thinking + answer
        input=[{"role": "developer", "content": SYSTEM_PROMPT},
               {"role": "user", "content": question}],
    )
    u = getattr(resp, "usage", None)
    det = getattr(u, "output_tokens_details", None) if u else None
    reasoning_tokens = (getattr(det, "reasoning_tokens", 0) or 0) if det else 0
    truncated = getattr(resp, "status", None) == "incomplete"
    reason = None
    if truncated:
        idet = getattr(resp, "incomplete_details", None)
        reason = getattr(idet, "reason", None) if idet else None
    return {"answer": resp.output_text or "",
            "latency_s": time.perf_counter() - t0,
            "in_tokens": getattr(u, "input_tokens", 0) or 0,
            "out_tokens": getattr(u, "output_tokens", 0) or 0,   # includes reasoning_tokens
            "reasoning_tokens": reasoning_tokens,
            "truncated": truncated,
            "error": (f"incomplete: {reason}" if truncated else None)}


# --- Anthropic Messages (claude-opus-5) --------------------------------------
#   * ADAPTIVE thinking + output_config.effort (NOT deprecated budget_tokens,
#     which returns 400 on Opus 5 / Sonnet 5 / 4.7+)
#   * no explicit temperature (Opus 5 rejects a non-default one)
#   * thinking_tokens read from usage details; truncation via stop_reason
def run_anthropic(client, model_id, question):
    t0 = time.perf_counter()
    resp = client.messages.create(
        model=model_id, max_tokens=2048,        # hard ceiling covers thinking + answer
        thinking={"type": "adaptive"},
        output_config={"effort": "low"},
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": question}],
    )
    text = "".join(b.text for b in resp.content if getattr(b, "type", None) == "text")
    u = getattr(resp, "usage", None)
    det = getattr(u, "output_tokens_details", None) if u else None
    thinking_tokens = (getattr(det, "thinking_tokens", 0) or 0) if det else 0
    stop = getattr(resp, "stop_reason", None)
    truncated = stop in ("max_tokens", "model_context_window_exceeded")
    return {"answer": text,
            "latency_s": time.perf_counter() - t0,
            "in_tokens": getattr(u, "input_tokens", 0) or 0,
            "out_tokens": getattr(u, "output_tokens", 0) or 0,   # includes thinking_tokens
            "reasoning_tokens": thinking_tokens,
            "truncated": truncated,
            "error": (f"incomplete: {stop}" if truncated else None)}


# --- OpenAI-compatible chat (DeepSeek) -- seed + temperature 0 ---------------
def run_chat(client, model_id, question):
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=model_id, temperature=0, seed=SEED, max_tokens=2048,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": question}],
    )
    choice = resp.choices[0]
    m = choice.message
    u = getattr(resp, "usage", None)
    truncated = getattr(choice, "finish_reason", None) == "length"
    return {"answer": m.content or "",
            "latency_s": time.perf_counter() - t0,
            "in_tokens": getattr(u, "prompt_tokens", 0) or 0,
            "out_tokens": getattr(u, "completion_tokens", 0) or 0,
            "reasoning_tokens": 0,               # DeepSeek chat exposes no separate breakdown
            "truncated": truncated,
            "error": ("incomplete: length" if truncated else None)}


_DISPATCH = {"openai_responses": run_openai, "anthropic": run_anthropic, "chat": run_chat}

def run_agent_live(model_key, question):
    spec = MODELS[model_key]
    try:
        return _DISPATCH[spec["kind"]](CLIENTS[model_key], spec["model_id"], question)
    except Exception as e:
        return {"answer": "", "latency_s": None, "in_tokens": 0, "out_tokens": 0,
                "reasoning_tokens": 0, "truncated": False,
                "error": f"{type(e).__name__}: {e}"[:300]}

print("Live adapters ready (developer msg + adaptive thinking + reasoning-token + truncation guard).")

## 6 · MOCK mode — deterministic, no API keys required

If a contender has **no key**, we synthesize its results from a **seeded RNG** (SHA-256 over `model_key | qid | field`). This makes the whole notebook run top-to-bottom and renders every chart, while staying perfectly reproducible.

The mock **profiles** below encode the *story* we're teaching: the cheap model is a hair behind the flagship on quality, much faster, and dramatically cheaper. They are **illustrative teaching data, not benchmark claims.**

In [ ]:
# Mock model profiles -- illustrative ONLY (teaching data, not a benchmark).
# reason_tok encodes the STORY that reasoning tokens are hidden output cost: the
# flagship "thinks" more (more thinking tokens -> more $), the cheap model less.
MOCK_PROFILES = {
    "deepseek-v4-pro": {"answer_quality": 0.90, "base_latency_s": 1.6, "tok_scale": 1.00, "reason_tok": 90},
    "claude-opus-5":   {"answer_quality": 0.93, "base_latency_s": 3.4, "tok_scale": 1.15, "reason_tok": 420},
    "gpt-5.6":         {"answer_quality": 0.93, "base_latency_s": 2.6, "tok_scale": 1.10, "reason_tok": 360},
}

def stable_random(*parts):
    """Deterministic float in [0,1) from a string key (sha256)."""
    key = "|".join(str(p) for p in parts).encode("utf-8")
    return int(hashlib.sha256(key).hexdigest()[:12], 16) / float(16 ** 12)

def estimate_tokens(text):
    return max(1, len(text) // 4)

def run_agent_mock(model_key, task):
    prof = MOCK_PROFILES.get(model_key, {"answer_quality": 0.85, "base_latency_s": 2.0,
                                         "tok_scale": 1.0, "reason_tok": 150})
    qid = task["id"]
    answer_ok = stable_random(model_key, qid, "answer") <= prof["answer_quality"]
    if answer_ok:
        answer = task["reference"]
    else:
        answer = task["reference"].split(".")[0] + ". (details omitted / partially correct)"
    # deterministic latency jitter around the base
    jitter = (stable_random(model_key, qid, "latency") - 0.5) * 0.8
    latency = round(prof["base_latency_s"] * (1 + jitter), 3)
    in_tokens = int((estimate_tokens(SYSTEM_PROMPT + task["question"]) + 40) * prof["tok_scale"])
    visible_out = int((estimate_tokens(answer) + 30) * prof["tok_scale"])
    # hidden reasoning/thinking tokens -- billed as OUTPUT tokens (deck S3)
    reason_jitter = 0.6 + stable_random(model_key, qid, "reason")          # 0.6 .. 1.6
    reasoning_tokens = int(prof["reason_tok"] * reason_jitter * prof["tok_scale"])
    out_tokens = visible_out + reasoning_tokens                            # cost bills both
    return {"answer": answer, "latency_s": latency,
            "in_tokens": in_tokens, "out_tokens": out_tokens,
            "reasoning_tokens": reasoning_tokens,
            "truncated": False, "error": None}


# Track contenders whose live calls fail so we can fall back to mock gracefully.
_LIVE_FALLBACK = set()

def run_agent(model_key, task):
    """Live if the key is present and working; otherwise deterministic mock.

    If a live contender errors (e.g. a placeholder / unavailable model id, or a
    truncated response with no answer), we fall back to the seeded mock for that
    model so the teaching narrative still runs end-to-end instead of collapsing
    to 0% accuracy.
    """
    if model_key in ACTIVE and model_key not in _LIVE_FALLBACK:
        out = run_agent_live(model_key, task["question"])
        if not out["error"] and out["answer"].strip():
            return out
        _LIVE_FALLBACK.add(model_key)
        print(f"  [warn] live calls for {MODELS[model_key]['label']} failed "
              f"({out['error']}); falling back to MOCK for this model.")
    return run_agent_mock(model_key, task)


# ---------------------------------------------------------------- mode policy
# Apples-to-apples rule: a live-vs-mock comparison is meaningless (a live model
# judged by a live judge vs deterministic mock data). So we only go LIVE when
# BOTH contenders have a working key; otherwise BOTH run in deterministic mock.
BOTH_HAVE_KEYS = all(k in ACTIVE for k in CONTENDERS)
FULLY_MOCK = not BOTH_HAVE_KEYS

# When not both keyed, force every contender down the mock path for consistency.
if FULLY_MOCK:
    ACTIVE = [k for k in ACTIVE if k not in CONTENDERS]

# ---------------------------------------------------------------- mode banner
print("=" * 60)
if FULLY_MOCK:
    missing = [MODELS[k]['label'] for k in CONTENDERS if k not in ACTIVE
               and not have(*MODELS[k]['keys'])]
    print("RUNNING IN FULL MOCK MODE (both contenders mocked).")
    if missing:
        print("  Reason: missing key(s) for:", missing)
    else:
        print("  Reason: not all contenders have a live key.")
    print("  All results are deterministic synthetic data from a seeded RNG,")
    print("  so the cheap-vs-flagship comparison stays apples-to-apples.")
else:
    print("RUNNING IN FULL LIVE MODE -- both contenders use real API calls.")
print("=" * 60)

## 7 · LLM-as-judge

The judge scores each answer **0/1** against the reference. The judge uses OpenAI if available, else Anthropic, else a keyword-overlap fallback — so judging works in mock mode too. Judge calls are pinned to `temperature=0`.

In [ ]:
# Strict 0/1 grader. Direct + goal/criteria-oriented -- NO "think step by step"
# (over-prompting CoT hurts reasoning models; research doc, deck S9).
JUDGE_INSTRUCTIONS = (
    "You are a strict grader. Given a QUESTION, a REFERENCE answer, and a CANDIDATE "
    "answer, return ONLY a JSON object: {\"correct\": 0 or 1}. Mark 1 only if the "
    "candidate captures the key facts of the reference. JSON only, no prose."
)

def _parse_json(text):
    text = text.strip()
    if "{" in text:
        text = text[text.index("{"): text.rindex("}") + 1]
    return json.loads(text)

def _keyword_correct(reference, answer):
    ref_words = {w.lower().strip(".,()") for w in reference.split() if len(w) > 4}
    ans_words = {w.lower().strip(".,()") for w in answer.split()}
    overlap = len(ref_words & ans_words) / max(1, len(ref_words))
    return int(overlap > 0.35)

def judge(question, reference, answer):
    if not answer or not answer.strip():
        return 0
    prompt = (f"QUESTION:\n{question}\n\nREFERENCE:\n{reference}\n\n"
              f"CANDIDATE:\n{answer}\n\nReturn the JSON now.")
    try:
        if have("OPENAI_API_KEY"):
            r = _openai.responses.create(
                model="gpt-5.6", reasoning={"effort": "low"},
                temperature=0, seed=SEED, max_output_tokens=512,
                input=[{"role": "developer", "content": JUDGE_INSTRUCTIONS},
                       {"role": "user", "content": prompt}])
            return int(_parse_json(r.output_text)["correct"])
        elif have("ANTHROPIC_API_KEY"):
            from anthropic import Anthropic
            r = Anthropic().messages.create(
                model="claude-opus-5", max_tokens=512,
                thinking={"type": "adaptive"}, output_config={"effort": "low"},
                system=JUDGE_INSTRUCTIONS,
                messages=[{"role": "user", "content": prompt}])
            txt = "".join(b.text for b in r.content if getattr(b, "type", None) == "text")
            return int(_parse_json(txt)["correct"])
        else:
            return _keyword_correct(reference, answer)
    except Exception:
        return _keyword_correct(reference, answer)

_judge_name = ("OpenAI" if have("OPENAI_API_KEY")
               else "Anthropic" if have("ANTHROPIC_API_KEY")
               else "keyword-fallback")
print(f"Judge ready (judge = {_judge_name})")

## 8 · Run the comparison

`2 contenders × 10 prompts`. We capture per-prompt: answer, correctness (judge), latency, token usage, and estimated cost. Results land in a tidy DataFrame and a CSV (`reproducibility_cheap_vs_flagship_results.csv`).

In [ ]:
rows = []
total = len(CONTENDERS) * len(EVAL_SET)
i = 0
for key in CONTENDERS:
    label = MODELS[key]["label"]
    for task in EVAL_SET:
        i += 1
        out = run_agent(key, task)
        # actual mode can differ from intent if a live model fell back to mock
        mode = "live" if (key in ACTIVE and key not in _LIVE_FALLBACK) else "mock"
        correct = judge(task["question"], task["reference"], out["answer"])
        # reasoning tokens are inside out_tokens -> estimate_cost bills them as output
        cost = estimate_cost(key, out["in_tokens"], out["out_tokens"])
        rows.append({
            "model": label, "model_key": key, "mode": mode, "qid": task["id"],
            "correct": correct,
            "latency_s": out["latency_s"],
            "in_tokens": out["in_tokens"], "out_tokens": out["out_tokens"],
            "reasoning_tokens": out.get("reasoning_tokens", 0),
            "truncated": out.get("truncated", False),
            "est_cost_usd": cost,
            "error": out["error"], "answer": out["answer"],
        })
        flag = "OK " if not out["error"] else "ERR"
        lat = "  ?  " if out["latency_s"] is None else f"{out['latency_s']:5.2f}s"
        print(f"[{i:>2}/{total}] {flag} [{mode}] {label:16s} {task['id']} "
              f"correct={correct} rtok={out.get('reasoning_tokens', 0):>4} {lat} ${cost:.5f}")

results = pd.DataFrame(rows)
results.to_csv("reproducibility_cheap_vs_flagship_results.csv", index=False)
print("-" * 60)
print("Saved -> reproducibility_cheap_vs_flagship_results.csv")
print("(reasoning_tokens are a subset of out_tokens -- hidden thinking, billed as output)")
results[["model", "mode", "qid", "correct", "latency_s", "reasoning_tokens", "est_cost_usd"]].head(20)

## 9 · Aggregate — accuracy, latency, cost, and the cost ratio

In [ ]:
agg = (results.groupby(["model_key", "model"])
       .agg(accuracy=("correct", "mean"),
            mean_latency_s=("latency_s", "mean"),
            mean_reasoning_tokens=("reasoning_tokens", "mean"),
            total_cost_usd=("est_cost_usd", "sum"),
            n=("correct", "size"))
       .reset_index())

# order: cheap first, flagship second (stable, for the narrative)
order = {CHEAP_KEY: 0, FLAGSHIP_KEY: 1}
agg["__o"] = agg["model_key"].map(order)
agg = agg.sort_values("__o").drop(columns="__o").reset_index(drop=True)

cheap = agg[agg.model_key == CHEAP_KEY].iloc[0]
flag  = agg[agg.model_key == FLAGSHIP_KEY].iloc[0]

agg_display = agg.copy()
agg_display["accuracy"] = (agg_display["accuracy"] * 100).round(1).astype(str) + "%"
agg_display["mean_latency_s"] = agg_display["mean_latency_s"].round(2)
agg_display["mean_reasoning_tokens"] = agg_display["mean_reasoning_tokens"].round(0).astype(int)
agg_display["total_cost_usd"] = agg_display["total_cost_usd"].round(5)
print("-" * 60)
print("Comparison summary (mean_reasoning_tokens = hidden thinking, billed as output)")
print("-" * 60)
display(agg_display[["model", "accuracy", "mean_latency_s",
                     "mean_reasoning_tokens", "total_cost_usd", "n"]])

# Robust ratios: guard every denominator so a degenerate run never prints nan.
acc_match  = (cheap["accuracy"] / flag["accuracy"] * 100) if flag["accuracy"] > 0 else float("inf")
cost_pct   = (cheap["total_cost_usd"] / flag["total_cost_usd"] * 100) if flag["total_cost_usd"] > 0 else float("inf")
cost_ratio = (flag["total_cost_usd"] / cheap["total_cost_usd"]) if cheap["total_cost_usd"] > 0 else float("inf")
print(f"Cost ratio (flagship / cheap): {cost_ratio:.1f}x more expensive for the flagship")
print(f"Flagship's hidden reasoning tokens/call: {flag['mean_reasoning_tokens']:.0f} "
      f"vs cheap's {cheap['mean_reasoning_tokens']:.0f} "
      f"-- a big slice of the flagship's cost is invisible thinking.")

## 10 · The headline

In [ ]:
print("=" * 60)
print("HEADLINE")
print("=" * 60)
print(f"{cheap['model']} matched {acc_match:.0f}% of {flag['model']}'s accuracy")
print(f"at ~{cost_pct:.0f}% of the cost  (the flagship cost {cost_ratio:.1f}x more).")
print("-" * 60)
print(f"  {cheap['model']:18s}  acc={cheap['accuracy']:.0%}  "
      f"cost=${cheap['total_cost_usd']:.5f}  lat={cheap['mean_latency_s']:.2f}s")
print(f"  {flag['model']:18s}  acc={flag['accuracy']:.0%}  "
      f"cost=${flag['total_cost_usd']:.5f}  lat={flag['mean_latency_s']:.2f}s")
print("=" * 60)
if FULLY_MOCK:
    print("(MOCK numbers -- deterministic teaching data. Add API keys for a live run.)")

## 11 · Visualizations

Two views: an **accuracy bar** (cheap vs flagship) and a **cost-vs-accuracy scatter** (the cheap model should sit far to the left — cheap — at nearly the same height).

In [ ]:
colors = {CHEAP_KEY: "#2563eb", FLAGSHIP_KEY: "#d97757"}
bar_colors = [colors.get(k, "#64748b") for k in agg["model_key"]]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# (1) accuracy bars -----------------------------------------------------------
axes[0].bar(agg["model"], agg["accuracy"], color=bar_colors)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("accuracy (fraction correct)")
axes[0].set_title("Accuracy: cheap vs flagship")
for x, v in enumerate(agg["accuracy"]):
    axes[0].text(x, v + 0.02, f"{v:.0%}", ha="center", fontsize=11)
axes[0].tick_params(axis="x", rotation=10)

# (2) cost vs accuracy scatter ------------------------------------------------
for _, row in agg.iterrows():
    axes[1].scatter(row["total_cost_usd"], row["accuracy"],
                    s=320, color=colors.get(row["model_key"], "#64748b"),
                    edgecolors="black", alpha=0.85, zorder=3)
    axes[1].annotate(row["model"], (row["total_cost_usd"], row["accuracy"]),
                     xytext=(8, -4), textcoords="offset points", fontsize=10)
axes[1].set_xlabel("total estimated cost for the eval set (USD)  ->  pricier")
axes[1].set_ylabel("accuracy  ->  better")
axes[1].set_title("Cost vs accuracy  (top-LEFT = cheap & accurate = win)")
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.05)

fig.suptitle(f"DeepSeek V4 Pro vs {flag['model']}"
             + ("   [MOCK DATA]" if FULLY_MOCK else "   [live/mixed]"),
             fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 12 · Reproducibility check (mock mode is bit-exact)

To make the "repeatable, not anecdotal" point concrete: re-running the **mock** pipeline produces **identical** results because everything is seeded. (A live run would be *close* but not identical — see §4.)

In [ ]:
def mock_run_signature():
    sig = []
    for key in CONTENDERS:
        for task in EVAL_SET:
            out = run_agent_mock(key, task)
            sig.append((key, task["id"], round(out["latency_s"], 3),
                        out["in_tokens"], out["out_tokens"]))
    return hashlib.sha256(repr(sig).encode()).hexdigest()[:16]

s1 = mock_run_signature()
s2 = mock_run_signature()
print("Mock run signature (pass 1):", s1)
print("Mock run signature (pass 2):", s2)
print("-" * 60)
print("Identical across re-runs?  ", s1 == s2,
      "  <- seeded mock is reproducible by construction")

## 13 · Recap & caveats

**Methodology (what made this a *disciplined* vibe-check):**
1. **A fixed, narrow task** with a small **fixed-order eval set** (10 prompts, sorted by id, never shuffled).
2. **An LLM-as-judge** with a strict 0/1 rubric, so "correct" isn't decided by gut feeling.
3. **Seeds + `temperature=0`** everywhere they're accepted (Python, NumPy, OpenAI/DeepSeek `seed=`), plus the judge pinned to `temperature=0`.
4. **An editable cost table** so the cost comparison is explicit and updatable.

**The cost takeaway:** on a *specific narrow task*, the cheap model (`deepseek-v4-pro`) matched the bulk of the flagship's accuracy at a small fraction of the cost. The right default is **per-task selection**, not "always reach for the most expensive model."

**Caveats — read before you trust a vibe-check:**
- Hosted models are **not bit-exact** even at `temperature=0` + seed (FP non-determinism, batching, MoE routing, silent model updates). Seeds *shrink* variance; they don't remove it.
- **10 prompts is tiny.** A real decision wants dozens-to-hundreds of cases, ideally with confidence intervals or repeated trials.
- **The judge is itself a model** and can be wrong or biased; spot-check its grades.
- **Cheap-matches-flagship is task-specific.** It holds for narrow, well-specified tasks; it often *fails* on long-horizon agentic or frontier-difficulty reasoning. Re-run this harness on **your** task before generalizing.
- The pricing here is a **placeholder** — replace `COST_PER_MTOK` with live numbers.

**To extend:** swap in your own prompts in `EVAL_SET`, add more contenders from `MODELS`, run multiple trials per prompt to estimate variance, and turn on real keys to replace the mock path.

## 14 · Engineering best-practices checklist (production wrapper)

The reproducibility experiment above answers *which model*. This section answers *how to call it in production* — the deck **Section 3** pre-flight checklist, as one small, MOCK-friendly helper. `production_reasoning_call` wraps a reasoning call with the five checks that bite most often:

1. **Right effort pinned** — hold one effort per cached conversation (changing it busts the cache).
2. **Token headroom reserved** — `max_output_tokens` must cover *thinking + answer* or you truncate mid-output and still pay for the thinking.
3. **Reasoning tokens instrumented** — log the hidden `reasoning_tokens` / `thinking_tokens` and their share of output cost.
4. **Incomplete / truncation handled** — detect `truncated` (OpenAI `status=="incomplete"`, Anthropic `stop_reason`, chat `finish_reason=="length"`) and, in prod, raise the ceiling and retry once.
5. **Structured-output guard** — never trust a strict-schema parse blindly; a truncated JSON body won't parse.

It reuses the same `run_agent` dispatch, so it runs live with a key or deterministic mock without one.

In [ ]:
def production_reasoning_call(model_key, question, reference="", *,
                              effort="low", max_output_tokens=2048):
    """A reasoning call wrapped with the production checklist (deck S3).

    Runs live if the model has a working key, else deterministic MOCK. Returns
    (answer, report) where report surfaces the hidden reasoning-token cost and
    the truncation/parse guards -- everything an on-call engineer wants logged.
    """
    task = {"id": "adhoc", "question": question, "reference": reference or question}

    # (1) effort pinned by caller + (2) headroom reserved -- both are inputs the
    #     real adapters already honor (reasoning.effort / output_config.effort,
    #     max_output_tokens / max_tokens). We echo them into the report.
    out = run_agent(model_key, task)                      # mock-friendly dispatch

    # (3) instrument the hidden thinking cost
    rtok = out.get("reasoning_tokens", 0)
    otok = out.get("out_tokens", 0)
    cost = estimate_cost(model_key, out.get("in_tokens", 0), otok)

    # (4) incomplete / truncation guard (prod policy: bump ceiling + retry once)
    status = "ok"
    if out.get("truncated") or not out["answer"].strip():
        status = "incomplete -> would raise max_output_tokens and retry once"

    # (5) structured-output guard: parse defensively, never blindly
    def safe_json(text):
        try:
            return _parse_json(text), None
        except Exception as e:
            return None, f"{type(e).__name__}: JSON parse failed (guarded, no crash)"
    _, parse_err = safe_json(out["answer"])               # answers here are prose, not JSON

    report = {
        "model": MODELS[model_key]["label"],
        "effort_pinned": effort,
        "headroom_max_out_tokens": max_output_tokens,
        "reasoning_tokens": rtok,
        "output_tokens": otok,
        "reasoning_share_of_output": round(rtok / otok, 3) if otok else 0.0,
        "est_cost_usd": round(cost, 6),
        "status": status,
        "json_guard": parse_err or "n/a (prose answer)",
    }
    return out["answer"], report


print("=" * 60)
print("PRODUCTION REASONING CALL -- checklist wrapper (deck S3)")
print("=" * 60)
for mk in CONTENDERS:
    ans, report = production_reasoning_call(
        mk, "In one sentence: why are reasoning tokens billed as output tokens?",
        reference=("Because the hidden thinking a reasoning model emits is generated "
                   "like any other output, providers meter and price it as output tokens."),
        effort="low", max_output_tokens=2048)
    print(f"[{report['model']}]")
    for k, v in report.items():
        if k == "model":
            continue
        print(f"    {k:26s}: {v}")
    print(f"    answer: {ans[:90]}...")
    print("-" * 60)
print("Checklist covered: effort pinned - token headroom - reasoning_tokens")
print("instrumented - incomplete handled - structured-output guard.")